# COLAB compatible clustering tool.



In [ ]:
repo = 'https://github.com/DariahBE/clustering_assisted_gui.git'
!git clone {repo}

In [ ]:
import os
import sys
import pandas as pd

sys.path.append('clustering_assisted_gui/utils')

In [ ]:
import gpu_manager  #hardware interaction
import callables as c   # getters/setters for pipeline
## multiple 'steps' each have their own layer ==> data excahnge using callables
from vectorization_layer import TransformerGUI
from dimred_layer import DimensionalityReductionGUI
from clustering_layer import Clustermachine
from noise_extension_layer import NoiseExtender
from dimviz import DimensionVisualizer
from cluster_inspector import ClusterInspectorGUI


In [ ]:
max_gpus = 1        #(INT): how many GPUS is this notebook allowed to use at most?

######### Leave this block of code as is: #########
gpu_manager.pick_gpu(1, 'auto', int(max_gpus))
os.environ["TOKENIZERS_PARALLELISM"] = "true"
devices = gpu_manager.pick_gpu(mode = 'report', verbosity=0)
if not bool(devices):
    device = 'cpu'
else:
    device = 'cuda'

print(f"Using {device}")

In [ ]:
TransformerGUI(
    config_path = "./clustering_assisted_gui/transformerconfig.json",
    device = device,
    on_result = c.receive_embeddings
).display()

In [ ]:
DimensionalityReductionGUI(
    c.fetch_vectors,
    config_path = "clustering_assisted_gui/dimredconfig.json",
    on_result = c.receive_reduced
).display()

In [ ]:
DimensionVisualizer(
    c.get_reduced,
    normalize=True
).display()

In [ ]:
Clustermachine(
    c.get_reduced,
    config_path = "clustering_assisted_gui/clusterconfig.json",
    on_result = c.receive_vectorlabels,
    stringgetter = c.get_input_list
).display()

In [ ]:
NoiseExtender(
    c.get_reduced,
    c.get_cluster_labels,
    "clustering_assisted_gui/noise_extension_methods.json",
    c.receive_denoised_results
).display()

In [ ]:
ClusterInspectorGUI(
    c.get_input_list,
    c.get_reduced,
    c.get_cluster_labels,
    c.get_denoised_labels,
    c.get_denoised_sources
).display()